## Question 1

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
spark = SparkSession.builder \
                    .master('local[*]') \
                    .appName('SparkSQL') \
                    .getOrCreate()

In [7]:
spark.version

'4.1.1'

## Question 2

In [1]:
!curl -L https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -O

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 24 67.8M   24 16.4M    0     0  17.8M      0  0:00:03 --:--:--  0:00:03 17.9M
 60 67.8M   60 40.9M    0     0  21.2M      0  0:00:03  0:00:01  0:00:02 21.3M
 96 67.8M   96 65.5M    0     0  22.4M      0  0:00:03  0:00:02  0:00:01 22.4M
100 67.8M  100 67.8M    0     0  22.8M      0  0:00:02  0:00:02 --:--:-- 22.8M


In [4]:
df = spark.read.parquet('data/raw/yellow/2025/')

In [5]:
df.repartition(4).write.parquet('data/pq/yellow')

## Question 3

In [10]:
df.filter(F.to_date(df.tpep_pickup_datetime) == '2025-11-15').count()

162604

## Question 4

In [5]:
df = spark.read.parquet('data/pq/yellow/')

In [ ]:
df = df.withColumn(
    "duration_hours", 
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

In [ ]:
max_duration = df.select(F.max("duration_hours")).collect()[0][0]

print(f"Longest trip: {max_duration} hours")

Longest trip: 90.64666666666666 hours


## Question 6

In [8]:
dfZones = spark.read.parquet('data/pq/zones/')

In [9]:
dfWithZones = df.join(dfZones, df.PULocationID == dfZones.LocationID)

pickupCounts = dfWithZones.groupBy("Zone") \
    .count() \
    .orderBy("count", ascending=True)

pickupCounts.show(5)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+
only showing top 5 rows
